# 🧹 Tiền xử lý Dữ liệu TFT

Notebook này thực hiện:
1. Nạp và kiểm tra dữ liệu thô
2. Làm sạch: missing values, outliers, duplicates
3. Gán nhãn Win/Lose (Top 4)
4. Tạo transaction format cho Association Rule Mining
5. Tạo co-occurrence matrix cho Graph Mining


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import warnings
warnings.filterwarnings('ignore')

from src.preprocessor import TFTPreprocessor

pp = TFTPreprocessor()
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Nạp dữ liệu thô

In [ ]:
df = pp.load_data('../data/raw/tft_matches.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Kiểm tra Missing Values

In [ ]:
# Đếm missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percent': missing_pct})
print(missing_df[missing_df['Count'] > 0] if missing.sum() > 0 else 'Không có missing values! ✅')

# Visualize missing
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df.isnull().T, cbar=True, yticklabels=True, ax=ax, cmap='YlOrRd')
ax.set_title('Missing Values Heatmap')
plt.tight_layout()
plt.show()

## 3. Làm sạch dữ liệu

In [ ]:
print(f'Trước khi clean: {len(df):,} records')
df = pp.clean_data(df)
print(f'Sau khi clean: {len(df):,} records')

In [ ]:
# Kiểm tra phân phối placement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['placement'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Phân phối Placement')
axes[0].set_xlabel('Placement')
axes[0].set_ylabel('Số lượng')

df['level'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Phân phối Level')
axes[1].set_xlabel('Level')
axes[1].set_ylabel('Số lượng')

plt.tight_layout()
plt.show()

## 4. Gán nhãn Win/Lose

In [ ]:
df = pp.label_win_lose(df, threshold=4)

# Visualize
fig, ax = plt.subplots(figsize=(6, 4))
df['result'].value_counts().plot(kind='pie', autopct='%1.1f%%', 
                                  colors=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_title('Win vs Lose Distribution (Top 4 = Win)')
ax.set_ylabel('')
plt.show()

## 5. Tạo Transaction Format (cho ARM)

In [ ]:
# Tạo transactions
transactions = pp.create_transactions(df, include_items=True, include_traits=True)

# Xem mẫu
print('\n📌 Mẫu transaction (Win):')
for t in list(transactions['win'])[:3]:
    print(f'  {set(t)}')

In [ ]:
# Tạo binary matrix
print('\n=== Binary Matrix cho Win ===')
bin_win = pp.create_binary_matrix(transactions['win'])
print(bin_win.head())

print('\n=== Binary Matrix cho Lose ===')
bin_lose = pp.create_binary_matrix(transactions['lose'])
print(bin_lose.head())

print('\n=== Binary Matrix cho All ===')
bin_all = pp.create_binary_matrix(transactions['all'])
print(bin_all.head())

## 6. Tạo Co-occurrence Matrix (cho Graph Mining)

In [ ]:
# Ma trận đồng xuất hiện theo tần suất
cooc_freq = pp.build_cooccurrence_matrix(df, entity='champions', weight='frequency')

# Ma trận đồng xuất hiện theo win-rate
cooc_wr = pp.build_cooccurrence_matrix(df, entity='champions', weight='winrate')

# Visualize top 20
top20 = cooc_freq.sum().nlargest(20).index
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

sns.heatmap(cooc_freq.loc[top20, top20], annot=False, cmap='YlOrRd', ax=axes[0])
axes[0].set_title('Co-occurrence (Frequency) - Top 20')

sns.heatmap(cooc_wr.loc[top20, top20], annot=False, cmap='RdYlGn', ax=axes[1])
axes[1].set_title('Co-occurrence (Win Rate) - Top 20')

plt.tight_layout()
plt.show()

## 7. Lưu dữ liệu đã xử lý

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

# Save processed dataframe
df.to_csv('../data/processed/tft_processed.csv', index=False)
print('✅ Saved: tft_processed.csv')

# Save binary matrices
bin_win.to_pickle('../data/processed/bin_win.pkl')
bin_lose.to_pickle('../data/processed/bin_lose.pkl')
bin_all.to_pickle('../data/processed/bin_all.pkl')
print('✅ Saved binary matrices')

# Save co-occurrence matrices
cooc_freq.to_pickle('../data/processed/cooc_frequency.pkl')
cooc_wr.to_pickle('../data/processed/cooc_winrate.pkl')
print('✅ Saved co-occurrence matrices')

# Summary
stats = pp.get_statistics(df)
print(f"\n📊 Tổng kết:")
print(f"   Records: {stats['total_records']:,}")
print(f"   Matches: {stats['total_matches']:,}")
print(f"   Champions: {stats['unique_champions']}")
print(f"   Items: {stats['unique_items']}")
print(f"   Win rate: {stats.get('win_rate', 0):.1%}")